# 00 · Setup

Creates the schemas and volumes, and a tiny table for the **day-one Power BI connector test**.
Run it once. Free Edition's default catalog is `workspace`; change `Config.catalog` if yours differs.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath("../src"))  # the repo's src/, via the Databricks Git folder

from favorita_spark.config import Config

cfg = Config()
spark.sql(f"USE CATALOG {cfg.catalog}")

In [ ]:
for schema in ("bronze", "silver", "gold"):
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema}")
spark.sql("CREATE VOLUME IF NOT EXISTS bronze.raw")     # upload the sliced CSVs here
spark.sql("CREATE VOLUME IF NOT EXISTS gold.export")    # Parquet + chart HTML for download
spark.sql("""
    CREATE OR REPLACE TABLE gold.connector_check AS
    SELECT 'hello from Databricks Free Edition' AS message, current_timestamp() AS created_at
""")
display(spark.sql("SHOW SCHEMAS"))

## Upload the raw slice

Locally: `python scripts/slice_raw.py --raw ../Flagship/data/raw --out data/slice`, then either

- **CLI:** `for f in data/slice/*.csv.gz; do databricks fs cp "$f" dbfs:/Volumes/workspace/bronze/raw/ --overwrite; done`, or
- **UI:** Catalog → `workspace` → `bronze` → `raw` → *Upload to this volume* (5 GB per file limit).
  Select the five files, not the `slice` folder, or they land in `raw/slice/`.

The next cell should list five `.csv.gz` files.

In [ ]:
display(dbutils.fs.ls(cfg.raw_path))

## Day-one test: can Power BI connect with a token?

The Free Edition docs don't say whether BI tools can log in with a personal access token, so test it before relying on it.

1. **SQL Warehouses** → the starter warehouse → **Connection details**: copy *Server hostname* and *HTTP path*.
2. Your user menu → **Settings → Developer → Access tokens → Generate new token** (short lifetime, e.g. 7 days).
3. Power BI Desktop → **Get data → Azure Databricks** → paste the hostname and HTTP path → *Import* →
   **Personal Access Token** → paste the token.
4. Navigate to `workspace` → `gold` → `connector_check` and load it.

Record the result in `docs/decisions/ADR-002-power-bi-link.md`. If it fails, notebook 07 exports Parquet for OneLake instead.